# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/SaimAli0001/Flyrank-Internship-ml/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

In [27]:
from dotenv import load_dotenv
import os
import duckdb
import pandas as pd

load_dotenv("../../.env")

token = os.getenv("HF_TOKEN")
print("Token loaded:", token is not None)

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE huggingface,
    TOKEN '{token}'
)
""")

print("DuckDB connection ready.")
def build_monthly_model_frame(feature_month, outcome_month):
    """
    Build one month-to-next-month modeling frame.

    feature_month: month used for features, e.g. '2026-03'
    outcome_month: immediately following month used for target, e.g. '2026-04'
    """

    feature_rel = f"""
    read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month={feature_month}/*.parquet'
    )
    """

    outcome_rel = f"""
    read_parquet(
        'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month={outcome_month}/*.parquet'
    )
    """

    query = f"""
    WITH feature_daily AS (
        SELECT
            report_date,
            client_hash_id,
            content_hash_id,
            gsc_impressions,
            gsc_clicks,
            gsc_avg_position
        FROM {feature_rel}
        WHERE gsc_data_available IS TRUE
    ),

    feature_monthly AS (
        SELECT
            client_hash_id,
            content_hash_id,

            SUM(gsc_impressions) AS impressions,

            CASE
                WHEN SUM(gsc_impressions) > 0
                THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
                ELSE NULL
            END AS ctr,

            AVG(gsc_avg_position) AS avg_position,

            AVG(
                CASE
                    WHEN EXTRACT(DAY FROM report_date) <= 15
                    THEN gsc_impressions
                END
            ) AS first_half_impressions,

            AVG(
                CASE
                    WHEN EXTRACT(DAY FROM report_date) > 15
                    THEN gsc_impressions
                END
            ) AS second_half_impressions,

            SUM(
                CASE
                    WHEN EXTRACT(DAY FROM report_date) <= 15
                    THEN gsc_clicks
                    ELSE 0
                END
            ) AS first_half_clicks,

            SUM(
                CASE
                    WHEN EXTRACT(DAY FROM report_date) <= 15
                    THEN gsc_impressions
                    ELSE 0
                END
            ) AS first_half_total_impressions,

            SUM(
                CASE
                    WHEN EXTRACT(DAY FROM report_date) > 15
                    THEN gsc_clicks
                    ELSE 0
                END
            ) AS second_half_clicks,

            SUM(
                CASE
                    WHEN EXTRACT(DAY FROM report_date) > 15
                    THEN gsc_impressions
                    ELSE 0
                END
            ) AS second_half_total_impressions

        FROM feature_daily

        GROUP BY
            client_hash_id,
            content_hash_id
    ),

    outcome_daily AS (
        SELECT
            client_hash_id,
            content_hash_id,
            gsc_impressions,
            gsc_clicks
        FROM {outcome_rel}
        WHERE gsc_data_available IS TRUE
    ),

    outcome_monthly AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) AS outcome_impressions,
            SUM(gsc_clicks) AS outcome_clicks,

            CASE
                WHEN SUM(gsc_impressions) > 0
                THEN SUM(gsc_clicks) * 1.0 / SUM(gsc_impressions)
                ELSE NULL
            END AS outcome_ctr

        FROM outcome_daily

        GROUP BY
            client_hash_id,
            content_hash_id
    )

    SELECT
        f.client_hash_id,
        f.content_hash_id,

        f.impressions AS march_impressions,
        f.ctr AS march_ctr,
        f.avg_position AS march_avg_position,

        f.second_half_impressions
            - f.first_half_impressions
            AS impression_trend,

        CASE
            WHEN f.first_half_total_impressions > 0
             AND f.second_half_total_impressions > 0
            THEN
                (f.second_half_clicks * 1.0
                 / f.second_half_total_impressions)
                -
                (f.first_half_clicks * 1.0
                 / f.first_half_total_impressions)
            ELSE NULL
        END AS ctr_trend,

        o.outcome_ctr,

        CASE
            WHEN o.outcome_ctr < f.ctr THEN 1
            ELSE 0
        END AS ctr_decline

    FROM feature_monthly f

    INNER JOIN outcome_monthly o
        USING (client_hash_id, content_hash_id)

    WHERE f.ctr IS NOT NULL
      AND o.outcome_ctr IS NOT NULL
    """

    return con.sql(query).df()

Token loaded: True
DuckDB connection ready.


In [28]:
mar_apr = build_monthly_model_frame("2026-03", "2026-04")

print("Mar → Apr rows:", len(mar_apr))

Mar → Apr rows: 158549


In [29]:
jan_feb = build_monthly_model_frame("2026-01", "2026-02")
feb_mar = build_monthly_model_frame("2026-02", "2026-03")

train_time = pd.concat(
    [jan_feb, feb_mar],
    ignore_index=True
)

print("Jan → Feb:", jan_feb.shape)
print("Feb → Mar:", feb_mar.shape)
print("Combined training rows:", len(train_time))

Jan → Feb: (110867, 9)
Feb → Mar: (134238, 9)
Combined training rows: 245105


In [30]:
FEATURES = [
    "march_impressions",
    "march_ctr",
    "march_avg_position",
    "impression_trend",
    "ctr_trend",
]

TARGET = "ctr_decline"

In [31]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

X_time_train = train_time[FEATURES].copy()
y_time_train = train_time[TARGET].copy()

X_time_test = mar_apr[FEATURES].copy()

time_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        random_state=42,
        max_iter=1000
    ))
])

time_model.fit(X_time_train, y_time_train)

time_prob = time_model.predict_proba(X_time_test)[:, 1]

print("Training rows:", len(X_time_train))
print("Test rows:", len(X_time_test))
print("Predictions:", len(time_prob))

Training rows: 245105
Test rows: 158549
Predictions: 158549


In [32]:
time_results = mar_apr[
    [
        "client_hash_id",
        "content_hash_id",
        "march_impressions",
        "march_ctr",
        "march_avg_position",
        "impression_trend",
        "ctr_trend",
        "ctr_decline"
    ]
].copy()

time_results["model_probability"] = time_prob

time_results = time_results.sort_values(
    by="model_probability",
    ascending=False
).reset_index(drop=True)

time_results.head(20)

,client_hash_id,content_hash_id,march_impressions,march_ctr,march_avg_position,impression_trend,ctr_trend,ctr_decline,model_probability
0,client_3ffa76342f366962,content_81433a2ecceb87d5,2.0,0.500000,5.000000,0.000000,1.000000,1,1.0
1,client_3ffa76342f366962,content_503e142f55d4bccd,6.0,0.166667,22.625000,-0.666667,-0.200000,1,1.0
2,client_73cda7b4e4f265ea,content_9184e15d4762f828,7.0,0.142857,5.714286,NaN,NaN,1,1.0
3,client_3ffa76342f366962,content_e6deac094719ecc8,5.0,0.200000,5.625000,0.500000,-0.500000,1,1.0
4,client_3ffa76342f366962,content_e0c131232f82dcfc,1.0,1.000000,0.000000,NaN,NaN,1,1.0
5,client_3ffa76342f366962,content_6f20ed787e68d8a6,1.0,1.000000,1.000000,NaN,NaN,1,1.0
6,client_3ffa76342f366962,content_16ddc64aabb449a8,1.0,1.000000,3.000000,NaN,NaN,1,1.0
7,client_3ffa76342f366962,content_12df780d7fc31c98,42.0,0.119048,4.495370,1.714286,0.151515,1,1.0
8,client_3197e6291363b4db,content_30d82334808be212,1.0,1.000000,0.000000,NaN,NaN,1,1.0
9,client_3ffa76342f366962,content_ad58a51dafdbda01,8.0,0.125000,39.050000,NaN,NaN,1,1.0


In [33]:
import numpy as np

queue = time_results.copy()

queue["reason_code"] = np.select(
    [
        (queue["march_impressions"] < 20),
        (
            (queue["march_impressions"] >= 20)
            & (queue["march_ctr"] < 0.005)
            & (queue["model_probability"] >= 0.80)
        ),
        (
            (queue["march_impressions"] >= 20)
            & (queue["ctr_trend"] < 0)
            & (queue["model_probability"] >= 0.80)
        ),
        (
            (queue["march_impressions"] >= 20)
            & (queue["model_probability"] >= 0.80)
        ),
    ],
    [
        "LOW_VOLUME_RISK",
        "STRONG_VISIBILITY_LOW_CTR",
        "TREND_DETERIORATION",
        "HIGH_DECLINE_RISK",
    ],
    default="INSUFFICIENT_EVIDENCE"
)

queue["action_label"] = np.select(
    [
        queue["reason_code"] == "LOW_VOLUME_RISK",
        queue["reason_code"] == "STRONG_VISIBILITY_LOW_CTR",
        queue["reason_code"] == "TREND_DETERIORATION",
        queue["reason_code"] == "HIGH_DECLINE_RISK",
    ],
    [
        "MONITOR_BEFORE_ACTION",
        "REVIEW_SNIPPET",
        "REVIEW_TREND",
        "REVIEW_CTR_SNIPPET",
    ],
    default="MONITOR"
)

queue = queue.sort_values(
    "model_probability",
    ascending=False
).reset_index(drop=True)

queue.head(20)

,client_hash_id,content_hash_id,march_impressions,march_ctr,march_avg_position,impression_trend,ctr_trend,ctr_decline,model_probability,reason_code,action_label
0,client_3ffa76342f366962,content_81433a2ecceb87d5,2.0,0.500000,5.000000,0.000000,1.000000,1,1.0,LOW_VOLUME_RISK,MONITOR_BEFORE_ACTION
1,client_3ffa76342f366962,content_503e142f55d4bccd,6.0,0.166667,22.625000,-0.666667,-0.200000,1,1.0,LOW_VOLUME_RISK,MONITOR_BEFORE_ACTION
2,client_73cda7b4e4f265ea,content_9184e15d4762f828,7.0,0.142857,5.714286,NaN,NaN,1,1.0,LOW_VOLUME_RISK,MONITOR_BEFORE_ACTION
3,client_3ffa76342f366962,content_e6deac094719ecc8,5.0,0.200000,5.625000,0.500000,-0.500000,1,1.0,LOW_VOLUME_RISK,MONITOR_BEFORE_ACTION
4,client_3ffa76342f366962,content_e0c131232f82dcfc,1.0,1.000000,0.000000,NaN,NaN,1,1.0,LOW_VOLUME_RISK,MONITOR_BEFORE_ACTION
5,client_3ffa76342f366962,content_6f20ed787e68d8a6,1.0,1.000000,1.000000,NaN,NaN,1,1.0,LOW_VOLUME_RISK,MONITOR_BEFORE_ACTION
6,client_3ffa76342f366962,content_16ddc64aabb449a8,1.0,1.000000,3.000000,NaN,NaN,1,1.0,LOW_VOLUME_RISK,MONITOR_BEFORE_ACTION
7,client_3ffa76342f366962,content_12df780d7fc31c98,42.0,0.119048,4.495370,1.714286,0.151515,1,1.0,HIGH_DECLINE_RISK,REVIEW_CTR_SNIPPET
8,client_3197e6291363b4db,content_30d82334808be212,1.0,1.000000,0.000000,NaN,NaN,1,1.0,LOW_VOLUME_RISK,MONITOR_BEFORE_ACTION
9,client_3ffa76342f366962,content_ad58a51dafdbda01,8.0,0.125000,39.050000,NaN,NaN,1,1.0,LOW_VOLUME_RISK,MONITOR_BEFORE_ACTION


### Reason codes and actions

| Reason code | Meaning | Suggested action |
|---|---|---|
| `HIGH_DECLINE_RISK` | The model ranks the page highly for future CTR decline. | `REVIEW_CTR_SNIPPET` |
| `LOW_VOLUME_RISK` | The page ranks highly but has very low impressions, making CTR changes less stable. | `MONITOR_BEFORE_ACTION` |
| `STRONG_VISIBILITY_LOW_CTR` | The page has meaningful visibility but relatively low CTR. | `REVIEW_SNIPPET` |
| `TREND_DETERIORATION` | The page has negative recent CTR movement and a high model ranking. | `REVIEW_TREND` |
| `INSUFFICIENT_EVIDENCE` | Search volume is too low to support a confident action. | `MONITOR` |

The model determines ranking order, while the reason code explains why a page was surfaced. These actions are recommendations for human review, not automatic content changes.

In [34]:
reason_counts = (
    queue["reason_code"]
    .value_counts()
    .rename_axis("reason_code")
    .reset_index(name="n")
)

action_counts = (
    queue["action_label"]
    .value_counts()
    .rename_axis("action_label")
    .reset_index(name="n")
)

reason_counts, action_counts

(                 reason_code       n
 0      INSUFFICIENT_EVIDENCE  119559
 1            LOW_VOLUME_RISK   28411
 2        TREND_DETERIORATION    5339
 3          HIGH_DECLINE_RISK    4520
 4  STRONG_VISIBILITY_LOW_CTR     720,
             action_label       n
 0                MONITOR  119559
 1  MONITOR_BEFORE_ACTION   28411
 2           REVIEW_TREND    5339
 3     REVIEW_CTR_SNIPPET    4520
 4         REVIEW_SNIPPET     720)

In [35]:
print("Total queued pages:", len(queue))
print("Low-volume share:", (queue["march_impressions"] < 20).mean())

Total queued pages: 158549
Low-volume share: 0.17919381389980385



### Queue summary

The validated model produces a ranked queue of 158,549 pages, but the playbook does not convert every high model score into an editing recommendation.

The largest group is `INSUFFICIENT_EVIDENCE` (119,559 pages), followed by `LOW_VOLUME_RISK` (28,411 pages). Review oriented actions are limited to pages with stronger supporting evidence: `TREND_DETERIORATION` (5,339), `HIGH_DECLINE_RISK` (4,520), and `STRONG_VISIBILITY_LOW_CTR` (720).

Only 17.92% of the queue falls below the 20 impression evidence threshold. This means the low-volume safeguard is active but does not dominate the entire queue.

The model provides the ranking signal, while the reason code and action label translate that signal into a human-review workflow.

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

### Intended use

This playbook is intended for content and SEO teams who need to prioritize which pages deserve human review first.

The model ranking helps order pages based on their observed March search-performance signals and their estimated likelihood of future CTR decline. Reason codes then translate the ranking into practical review actions.

The intended workflow is:

1. Use the ranked queue to identify pages for attention.
2. Read the reason code and supporting metrics.
3. Review the page and its search context manually.
4. Decide whether a content or snippet change is appropriate.
5. Monitor the page after any approved change.

### Limits

The model is not intended to automatically rewrite, publish, remove, or refresh content.

The results are based on the available FlyRank dataset and the validation designs tested in Weeks 5 and 6. The stronger Week-5 performance did not fully carry over to the stricter time-forward validation, so future performance should not be assumed to match the Week-5 results.

Low-impression pages require additional caution because small numbers of impressions or clicks can produce unstable CTR changes. These pages are routed to monitoring rather than immediate action.

The model identifies pages that look worth reviewing. It does not establish that changing a page will improve CTR, rankings, traffic, or any other business outcome.

In [36]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

### Human review rules

Every page receiving a review-oriented action must be checked by a human before any change is made.

The reviewer should:

1. Confirm that the page has enough search evidence to justify action.
2. Check the current page and its search-result context.
3. Review the reason code and supporting March metrics.
4. Check whether the observed issue is still relevant before making a change.
5. Decide whether a snippet, content, or monitoring action is appropriate.
6. Record the decision and monitor the later outcome when possible.

A high model score is a reason to review a page, not a reason to automatically change it.

### No-go list

The system must not automatically:

- publish, rewrite, or delete content;
- change titles, descriptions, or other page elements without human approval;
- declare that a content change will improve CTR, rankings, traffic, or business results;
- act on pages with insufficient evidence simply because the model score is high;
- treat the model score as a replacement for editorial or SEO judgment;
- make decisions about new pages or tracking artifacts that do not have sufficient historical evidence.

### Human decision rule

The playbook should be treated as a prioritization aid:

> **Model ranks → reason code explains → human reviews → human decides → outcome is monitored.**

In [37]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

The playbook should be monitored on two clocks.

### Pre-release checks

Before generating a new queue, check:

- The expected feature columns are present.
- Feature distributions have not changed substantially from the data used to train the model.
- Missingness in the trend features remains within a reasonable range.
- The expected client/content grain is still valid.
- The model produces probabilities and rankings without errors.

A meaningful change in the feature distribution or missingness pattern should trigger an investigation before the queue is used.

### Post-label checks

After the future outcome window becomes available, evaluate:

- Precision@K at the same review depths used during validation.
- The observed decline rate / base rate.
- Whether the reason-code groups still show useful differences.
- Whether error patterns have materially changed.

A sustained drop in Precision@K or a large shift in the target base rate should trigger a review of the model and playbook.

### Retrain triggers

Retraining should be considered when:

1. Performance remains materially below the validated time-forward result over repeated outcome windows.
2. Feature distributions or missingness change enough to make the original training population no longer representative.
3. The business/data pipeline changes the meaning, availability, or calculation of a model feature.
4. The reason-code groups stop separating pages in a useful way.

Retraining should be evaluated and validated under the same honest split design before a new model replaces the existing one.

The monitoring process is intended to detect when the recommendations may have become stale. It does not assume that every change requires immediate retraining.

In [38]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

The final playbook queue is exported to `work/outputs/` so that the research paper can use the same ranked output produced by this notebook.

The exported queue contains the pseudonymous content identifier, model ranking, supporting March metrics, reason code, and suggested action.

The queue is regenerated by the notebook and is not treated as a separately maintained data artifact.

A small summary of the queue is also prepared for reuse in the paper, including the number of pages in each reason-code and action group.

In [39]:
from pathlib import Path

output_dir = Path("../../work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

# Public-safe queue: exclude client identifier and observed future target
queue_export = queue[
    [
        "content_hash_id",
        "march_impressions",
        "march_ctr",
        "march_avg_position",
        "impression_trend",
        "ctr_trend",
        "model_probability",
        "reason_code",
        "action_label",
    ]
].copy()

queue_export.insert(
    0,
    "rank",
    range(1, len(queue_export) + 1)
)

queue_path = output_dir / "w07_action_queue.csv"
queue_export.to_csv(queue_path, index=False)

summary = (
    queue_export["reason_code"]
    .value_counts()
    .rename_axis("reason_code")
    .reset_index(name="n")
)

summary_path = output_dir / "w07_reason_code_summary.csv"
summary.to_csv(summary_path, index=False)

print("Queue exported to:", queue_path)
print("Summary exported to:", summary_path)
print("Queue rows:", len(queue_export))

Queue exported to: ..\..\work\outputs\w07_action_queue.csv
Summary exported to: ..\..\work\outputs\w07_reason_code_summary.csv
Queue rows: 158549


In [40]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print(queue_export.head(10))
print("\nReason-code summary:")
print(summary)

   rank           content_hash_id  march_impressions  march_ctr  \
0     1  content_81433a2ecceb87d5                2.0   0.500000   
1     2  content_503e142f55d4bccd                6.0   0.166667   
2     3  content_9184e15d4762f828                7.0   0.142857   
3     4  content_e6deac094719ecc8                5.0   0.200000   
4     5  content_e0c131232f82dcfc                1.0   1.000000   
5     6  content_6f20ed787e68d8a6                1.0   1.000000   
6     7  content_16ddc64aabb449a8                1.0   1.000000   
7     8  content_12df780d7fc31c98               42.0   0.119048   
8     9  content_30d82334808be212                1.0   1.000000   
9    10  content_ad58a51dafdbda01                8.0   0.125000   

   march_avg_position  impression_trend  ctr_trend  model_probability  \
0            5.000000          0.000000   1.000000                1.0   
1           22.625000         -0.666667  -0.200000                1.0   
2            5.714286               NaN    

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.